# Programming for Data Science — Assignment 07
## Assignment 07: Neural Networks as Python Classes
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Assignments/Assignment_07_Neural_Network_Classes.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**

This assignment goes with the in-class notebook **`11_Classes_OOP.ipynb`**. First you walk through the whole roadmap of a neural-network project on a tiny network, from loading the data to a final test. Then you write a network of your own as a Python class, by filling in a template, and explain in your own words what the class does and what the network learned.

About this notebook:  
The design by Chenguang Pan, in collaboration with AI. I am responsible for its accuracy.

---
### 📋 How this assignment works (Read me carefully)

- **Run on Google Colab.** Open this notebook with the badge above, then *File ▸ Save a copy in Drive* so you have your own copy to edit. PyTorch, the library we use for the networks, comes installed in Colab, so there is nothing to install.
- **The data.** The same two files as in Assignment 06: 4,424 students at a college in Portugal, split at random into a training file and a test file with 2,212 students each. This time a fifth of the training file is set aside as validation students, for checking a network while you work on it. The test file is used once, at the very end.
- **The question.** Also the same: at the end of the first semester, who is likely to drop out? Every model answers 1 ("likely to drop out") or 0 for each student.
- **What is new.** A neural network. In PyTorch every network is a Python class, so this week the classes from Session 11 go to work. You do not need any math or statistics: the notebook explains each idea in plain words, and the main number you read is accuracy, the share of students a model guesses right.
- **Three parts.** Part 1 runs the whole roadmap, all ten steps, on a tiny network. You read and run it, and answer three short questions. In Part 2 you write a bigger network as a class of your own, by filling in a template; the training and the plot are done for you. In Part 3 you put the two networks side by side and explain, in your own words, what your class does and what the networks learned.
- **AI help.** Try each part on your own first. After that you may ask an AI tool to explain an error or a concept, as the syllabus describes. Name the tool and what you used it for in `ai_note`, in the cell under Before you start. Before you ask an AI tool, download your notebook and keep that copy as your draft; the Submission section says what to upload.
- **Handle with care.** These are real, anonymized students. When a model flags someone, it means an advisor should check in.
- Give your class a **docstring**, and drop a **comment** on each line you add or change.

**Total: 100 points** — Part 1: 20 · Part 2: 40 · Part 3: 30 · Code quality: 10.  

Coding load is light for this assignment, but the thinking and understanding load is heavier. 

---
### ✍️ Before you start
Put your name in the next cell, and write down any AI tool you used (leave it empty if you used none). Then run the setup cell right after it.

In [ ]:
student_name = ""   # <-- type your name between the quotes
ai_note = ""        # <-- e.g. "ChatGPT: explained a shape error in Part 2" (leave "" if none)
print("Submission by:", student_name)

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch                          # <-- a famous package for modern AI
from torch import nn                  # nn holds PyTorch's building blocks for neural networks

# The same two files as in Assignment 06, read straight from the public course repo.
TRAIN_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/dropout_train.csv"
TEST_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/dropout_test.csv"

print("pandas", pd.__version__, "| PyTorch", torch.__version__)
print("Setup complete ✅")

---
## Part 1 — The roadmap, on a tiny network (read and answer the questions, no coding. 20 points)
  
**Why neural networks?**  
Neural networks (NNs) are the most flexible and powerful models in machine learning. Theoretically, when data is abundant, a NN can learn any statistical pattern. That is why people use it for prediction or any fundamental task in modern AI, like Generative Adversarial Networks (GANs), diffusion models, reinforcement learning, or Large Language Models (LLMs).  

I intentionally design this assignment to guide you through the basic knowledge of neural networks. You do not need to understand every technical detail behind the NN. The overall key takeaway for this assignment is: **when you are handling complex data, say thousands of students with 10 or more background variables, and you want to build a model with excellent predictive performance, try NN, although it might not be the best model.**

Every network in this assignment goes through the ten steps below. Part 1 runs all ten on a tiny network, so that you see each step once before you make a network of your own. Read each cell before you run it. You do not need to change anything here; the only writing is three short answers at the end.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_roadmap.png" width="800" alt="The ten steps of the roadmap, with the three groups of students each step uses">
</p>

Figure source: Made with Claude.

### Steps 1 and 2: load and prepare

The next cell reads both files, cleans the column names with `snake_case` from Assignment 06, and keeps the same 12 predictors you could use there: every column except `student_id`, `course`, and `target`. A network needs numbers, so `gender` becomes 1 for male and 0 for female, and `daytime_evening_attendance` becomes 1 for evening classes and 0 for daytime. The outcome is 1 for a student who dropped out and 0 for everyone else.

Please read the following cell and try to understand it. You do not need to change anything here. 

In [ ]:
# Steps 1 and 2: load the two files and prepare them — run me, do not edit
def snake_case(name):
    """Return name in lowercase, with every run of other characters turned into one underscore (Assignment 06)."""
    return "_".join("".join(ch if ch.isalnum() else " " for ch in name.lower()).split())

def prepare(df):
    """Return the 12 predictors as numbers and the 0/1 dropout outcome for the students in df."""
    df = df.rename(columns={col: snake_case(col) for col in df.columns})
    X = df.drop(columns=["student_id", "course", "target"])
    X["gender"] = (X["gender"] == "male").astype(int)                                          # 1 = male
    X["daytime_evening_attendance"] = (X["daytime_evening_attendance"] == "evening").astype(int)   # 1 = evening
    y = (df["target"] == "Dropout").astype(int)                                                # 1 = dropped out
    return X, y

X_file, y_file = prepare(pd.read_csv(TRAIN_URL))                 # steps 1 and 2 for the training file
X_test_table, y_test_table = prepare(pd.read_csv(TEST_URL))      # and for the test file

print("the 12 predictors:", list(X_file.columns))

### Step 3: set aside the validation students

When we are training a NN, we need to monitor and evaluate the fitting progress to see whether it works well on an unseen dataset; this unseen dataset is called the validation set. After finishing the training, we test our fitted model on the other unseen dataset, which we call the test set. Therefore, we need to separate the observed data into three pieces: the training set, the validation set, and the test set.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_train_valid_test.png" width="800" alt="The training file split into training and validation sets, the test file kept apart, and what each set is for">
</p>

Figure source: Made with Claude.

In [ ]:
# Step 3: hold out 20% of the training file as validation students — run me, do not edit
valid_rows = X_file.sample(frac=0.2, random_state=5001).index        # the same students for everyone
X_valid_table, y_valid_table = X_file.loc[valid_rows], y_file.loc[valid_rows]
X_train_table, y_train_table = X_file.drop(index=valid_rows), y_file.drop(index=valid_rows)

print("training students:", len(X_train_table), "| validation students:", len(X_valid_table),
      "| test students:", len(X_test_table))

### Steps 4 and 5: scale, then turn the tables into tensors

Admission grades run from about 95 to 190, while most other columns are just 0 or 1, and a network learns more smoothly when its inputs are on similar scales. So within each column, a value becomes a z-score: how far that student sits from the training students' average on that column, counted in standard deviations, where a standard deviation is the typical distance from the average. A z-score of 0 is average, and 1 is one typical distance above it.

The averages and standard deviations come from the training students only, and the validation and test students are rescaled with those same numbers. Like the norms in norm-referenced scoring, they are set once and used for everyone who comes later. It is the idea behind the `Analysis` class in the Session 11 slides. Finally the tables become tensors, PyTorch's version of a NumPy array.

In [ ]:
# Steps 4 and 5: scale with the training students' numbers, then make tensors — run me, do not edit
train_mean = X_train_table.mean()         # learned from the training students only
train_sd = X_train_table.std()

# converting the input numbers into a tensor is a critical step for efficient computation
def to_tensor(table):
    """Return a pandas table or Series as a PyTorch tensor of decimal numbers."""
    return torch.tensor(table.to_numpy(), dtype=torch.float32)

X_train = to_tensor((X_train_table - train_mean) / train_sd)
X_valid = to_tensor((X_valid_table - train_mean) / train_sd)     # the training students' numbers,
X_test = to_tensor((X_test_table - train_mean) / train_sd)       # used for every group
y_train, y_valid, y_test = to_tensor(y_train_table), to_tensor(y_valid_table), to_tensor(y_test_table)

print("X_train:", tuple(X_train.shape), "| X_valid:", tuple(X_valid.shape), "| X_test:", tuple(X_test.shape))

### Step 6: build the network as a class

Think of a network as layers of small calculators, called units. Each unit in a hidden layer does two simple things.

1. It takes a weighted sum of the numbers that come into it, plus a constant. The weights say how much each input counts, and the network learns them from the training students.
2. It passes the sum through a bend called ReLU, which keeps positive numbers and turns negative ones into 0. Without the bend, a stack of layers would collapse into one big weighted sum.

A student's 12 numbers go into the first hidden layer. Its units hand their results to the last layer, which has a single unit and gives one risk score per student. That unit only takes the weighted sum, with no bend, so the risk score can be any number, negative or positive. Later, `torch.sigmoid` squeezes each risk score into a probability between 0 and 1 (a risk score below 0 becomes a probability below one half), and the guess is 1 ("likely to drop out") when the probability is at least 0.5, the same cutoff as in Assignment 06. So in every network you write, the last layer gets no bend and no squeeze. A network with no hidden layer is exactly the logistic regression from Assignment 06: one weighted sum, squeezed into a probability.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_anatomy.png" width="800" alt="TinyNet: 12 inputs, 8 hidden units with the ReLU bend, and one output risk score">
</p>

Figure source: Made with Claude.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_one_unit.png" width="800" alt="One hidden unit: inputs times weights, plus a constant, then the ReLU bend">
</p>

Figure source: Made with Claude.

Words you will meet in this assignment:

| Word | What it means here |
|---|---|
| unit | one small calculator: a weighted sum of its inputs plus a constant, followed by the bend in a hidden layer |
| layer | a group of units that all read the same inputs. `nn.Linear(12, 8)` is a layer that reads 12 numbers and has 8 units |
| hidden layer | any layer between the inputs and the last layer |
| weight | a number the network learns, saying how much one input counts for one unit. The weight counts in this notebook include the constants too |
| ReLU | the bend: keeps positive numbers, turns negative ones into 0 |
| risk score, probability | the last layer gives a risk score, which can be any number; `torch.sigmoid` squeezes it into a probability between 0 and 1 |
| loss | one number that says how far the network's probabilities are from what really happened; training makes it small |
| optimizer | the rule that nudges the weights to make the loss smaller; the learning rate sets the size of each nudge |
| epoch | one round in which the network looks at all the training students and nudges its weights once |
| overfitting | memorizing the training students instead of learning patterns that also hold for new students |

Here is the network from the pictures, written as a class. Read it the way you read `GradStudent` in Session 11. If we skipped section 2.9 of the class notebook (inheritance) in class, read it before you go on.

In [ ]:
# Step 6: the network as a class — run me, do not edit
class TinyNet(nn.Module):                         # <---- inherits from PyTorch's Module class
    """A small network: 12 inputs, one hidden layer of 8 units, one output risk score."""

    def __init__(self, n_inputs=12, n_hidden=8):
        super().__init__()                        # <---- let nn.Module set itself up first
        self.fc1 = nn.Linear(n_inputs, n_hidden)  # <---- the hidden layer: 12 numbers in, 8 units
        self.fc2 = nn.Linear(n_hidden, 1)         # <---- the last layer: 8 numbers in, 1 risk score out

    def forward(self, x):
        """Send a batch of students x through the layers and return one risk score per student."""
        x = torch.relu(self.fc1(x))               # <---- the hidden layer, then the bend
        return self.fc2(x)                        # <---- no bend and no squeeze after the last layer

torch.manual_seed(5001)          # the same random starting weights every time you run this cell
tiny = TinyNet()                 # <---- one object built from the class
print(tiny)                      # nn.Module's __repr__ lists the layers

Everything here comes from Session 11, and four ideas from it carry this whole assignment.

- `TinyNet` is the class, a blueprint. `tiny = TinyNet()` builds one object from it, the way `ann = Student("Ann", [90, 85])` built a student. You can build many objects from one class, and each one gets its own weights.
- `class TinyNet(nn.Module):` makes PyTorch's `nn.Module` the parent class, the way `GradStudent(Student)` did. `TinyNet` inherits everything a network needs, such as keeping track of its weights and printing itself, and adds only its layers and its `forward` method. `super().__init__()` runs the parent's setup first, as in `GradStudent`. Online you will often see the older spelling `super(TinyNet, self).__init__()`, which does the same thing.
- `self.fc1` and `self.fc2` are attributes, and each one holds a layer, with its weights inside. (`fc` stands for "fully connected", a common name for this kind of layer.)
- `forward` is a method. It says how a student's numbers flow from layer to layer. You never call it by name: writing `tiny(X)` runs it for you.

`torch.manual_seed(5001)` fixes the random starting weights, so you get the same numbers every time you run the cell. A different seed, or a different version of PyTorch, gives slightly different numbers.

### Steps 7 and 8: choose the loss and the optimizer, then train

Training needs two more objects. The loss function turns the network's risk scores into one number that says how far off the guesses are. `nn.BCEWithLogitsLoss` is the usual choice for a yes/no outcome: it squeezes each risk score into a probability itself, charges little when the network is confident and right, and charges a lot when it is confident and wrong. The optimizer is the rule that nudges the weights to make the loss smaller. `torch.optim.Adam` is a popular one, and its learning rate, `lr`, sets how big each nudge is. The optimizer is built from `net.parameters()`, the weights inside the network object, so it knows which numbers it may change.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/nn_training_loop.png" width="800" alt="One epoch of training: forward, loss, backward, step">
</p>

Figure source: Made with Claude.

Training repeats four steps, once per epoch, on the training students only:

1. **Forward.** The network computes a risk score for every training student.
2. **Loss.** The loss function compares those risk scores with what really happened.
3. **Backward.** `loss.backward()` works out, for every weight, whether nudging it up or down would make the loss smaller.
4. **Step.** `optimizer.step()` nudges every weight a little in that direction.

To watch the network learn, the loop writes down two accuracies every 10 epochs, one for the training students and one for the validation students. The first cell below defines two small helpers for that. The second cell puts steps 7 and 8 into one function, `train_network`: you give it a network object, and it trains that object for 500 epochs and returns what it wrote down. Part 2 uses the same function for your network, so you never have to copy the loop.

In [ ]:
# Two small helpers for measuring accuracy — run me, do not edit
def guess(net, X):
    """Return a 0/1 guess per student: 1 when the network's probability of dropping out is at least 0.5."""
    net.eval()                                          # guessing mode, the opposite of net.train() in training
    with torch.no_grad():                               # only guessing, so PyTorch skips the training bookkeeping
        risk_scores = net(X).squeeze(1)                 # one risk score per student, as a flat list
        probability = torch.sigmoid(risk_scores)        # the squeeze: any number becomes a probability between 0 and 1
    return (probability >= 0.5).float()

def accuracy(net, X, y):
    """Return the share of students whose guess matches what happened, rounded to 3 decimals."""
    return round(float((guess(net, X) == y).float().mean()), 3)

In [ ]:
# Steps 7 and 8: the loss, the optimizer, and the training loop, in one function — run me, do not edit
def train_network(net, epochs=500):
    """Train the network object net on the training students; return the accuracies written down every 10 epochs."""
    loss_fn = nn.BCEWithLogitsLoss()                          # step 7: how far off the risk scores are, for a yes/no outcome
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)   # step 7: nudges the weights inside net; lr = learning rate
    history = {"epoch": [], "train_accuracy": [], "valid_accuracy": []}
    for epoch in range(1, epochs + 1):                        # step 8: one round per epoch
        net.train()                                           # training mode, the opposite of net.eval() in guess
        optimizer.zero_grad()                                 # forget the nudges from the last epoch
        risk_scores = net(X_train).squeeze(1)                 # 1. forward: one risk score per training student
        loss = loss_fn(risk_scores, y_train)                  # 2. loss: how far off, as one number
        loss.backward()                                       # 3. backward: which way to nudge each weight
        optimizer.step()                                      # 4. step: nudge every weight a little
        if epoch % 10 == 0:                                   # every 10 epochs, write down both accuracies
            history["epoch"].append(epoch)
            history["train_accuracy"].append(accuracy(net, X_train, y_train))
            history["valid_accuracy"].append(accuracy(net, X_valid, y_valid))
    print("loss after the last epoch:", round(loss.item(), 3))
    return history

history = train_network(tiny)        # <---- the object tiny goes in; its training record comes out

### Step 9: validate

The next cell draws the two accuracies that `train_network` wrote down. If the training line keeps climbing while the validation line stays flat or slides down, the network is memorizing the training students. That is overfitting. When you try different settings for a network, this is the step where you compare them: change a setting, go back to step 6, and train again. In Part 2 you build a bigger network and compare its plot with this one.

In [ ]:
# Step 9: compare the training and the validation students — run me, do not edit
def plot_history(history, title):
    """Draw the accuracy on the training and the validation students as training goes on."""
    if not history["epoch"]:
        print("Train for at least 10 epochs to get a plot.")
        return
    nobody = float((y_valid == 0).float().mean())      # the accuracy of guessing "nobody drops out"
    lowest = min(history["train_accuracy"] + history["valid_accuracy"])
    plt.figure(figsize=(7, 3.5))
    plt.plot(history["epoch"], history["train_accuracy"], label="training students")
    plt.plot(history["epoch"], history["valid_accuracy"], label="validation students")
    plt.axhline(nobody, color="gray", linestyle=":", label=f'"nobody drops out" ({nobody:.3f})')
    plt.ylim(min(0.6, lowest - 0.02), 1.0)              # the same scale for every plot, unless a line falls lower
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title)
    plt.legend()
    plt.show()
    if history["valid_accuracy"][-1] < nobody:
        print("⚠️ The validation accuracy ended below guessing that nobody drops out. Check that forward has "
              "no bend and no torch.sigmoid after the last layer.")

plot_history(history, "TinyNet: one hidden layer of 8 units")

### Step 10: test once

The final check uses the test students, whom no step so far has touched. `report` collects a network's numbers in a dictionary: how many weights it learned, and its accuracy on the training, validation, and test students. You will use it again in Part 3. Here it runs once on `tiny`, so you can see what it returns.

In [ ]:
# Step 10: the final check on the test students — run me, do not edit
def report(name, net):
    """Return a network's numbers in a dictionary: how many weights it learned, and its accuracy on each group."""
    return {"model": name,
            "weights": sum(p.numel() for p in net.parameters()),     # how many numbers the network learned
            "train_accuracy": accuracy(net, X_train, y_train),
            "valid_accuracy": accuracy(net, X_valid, y_valid),
            "test_accuracy": accuracy(net, X_test, y_test)}

tiny_results = report("TinyNet", tiny)
print(tiny_results)

**Your turn (20 points).** The cell below prints what question 1 asks about, so there is no code to write. Answer all three questions in a sentence or two each in the text cell under it.
1. `tiny.fc1` is an attribute that holds a layer, and the layer has attributes of its own: `weight` holds its weights, and `bias` holds its constants. What do the shapes printed below mean, and how do they add up to the 113 weights in `tiny_results`?
2. In the plot from step 9, how do the two lines move as training goes on? Is there a big gap between the training and the validation students at the end?
3. Why does the roadmap keep the test students out of steps 8 and 9? What would go wrong if you kept changing a network until it looked good on the test students?

In [ ]:
# Question 1: the layers stored inside tiny, and the weights stored inside each layer — run me, do not edit
print("tiny.fc1.weight:", tuple(tiny.fc1.weight.shape), "   tiny.fc1.bias:", tuple(tiny.fc1.bias.shape))
print("tiny.fc2.weight:", tuple(tiny.fc2.weight.shape), "   tiny.fc2.bias:", tuple(tiny.fc2.bias.shape))

*Double-click this cell and write your answers here.*

---
## Part 2 — Your own network, written as a class (40 points)

Now you write a network of your own and send it through the same roadmap. Steps 1 to 5 are done: your network learns from the same students. Steps 7 to 10 are the Part 1 functions `train_network`, `plot_history`, and `report`, and they work for any network object. So your only job is step 6, the class.  
(a) In the first cell below, fill in the template. `MyNet` is `TinyNet` with a second hidden layer. Replace `units_1` and `units_2` with the number of units you want in each hidden layer, for example 64 and 16. Replace `___` with a one-sentence docstring, and write the one line that `forward` is missing.  
(b) Run the second cell. It builds one object from your class, trains it, and plots it.  
(c) Run the checkpoint. If you change your class later, run (a), (b), and the checkpoint again: (b) builds a fresh `my_net` from the new class.

Two rules keep the layers fitting together. Each layer takes in as many numbers as the layer before it gives out: in TinyNet, `fc1` gives out 8 and `fc2` takes in 8. And the last layer gives out 1 number, the risk score.

🧰 **When PyTorch complains**

| The error ends with | What it usually means |
|---|---|
| `name 'units_1' is not defined` | a placeholder from the template is still there: write the number of units |
| `mat1 and mat2 shapes cannot be multiplied (1770x64 and 16x1)` | two layers do not fit together, or `forward` skips a layer: here a layer gives out 64 numbers and the next one expects 16 |
| `name 'fc2' is not defined` | in `forward`, a layer is written without `self.`: write `self.fc2` |
| `must be Tensor, not Linear` | a layer is used without `(x)`: write `self.fc2(x)` |

One mistake gives no error at all: if the validation accuracy comes out at about 0.35, the network guesses 1 for every student. Remove any bend or `torch.sigmoid` after the last layer.

In [ ]:
# Part 2 (a): your class. Fill in units_1, units_2, the docstring, and the missing line in forward
class MyNet(nn.Module):                          # <---- inherits from nn.Module, like TinyNet
    """___"""                                    # <-- replace ___ with one sentence on your network's shape

    def __init__(self, n_inputs=12):
        super().__init__()                       # <---- let nn.Module set itself up first
        self.fc1 = nn.Linear(n_inputs, units_1)  # <-- the first hidden layer: 12 numbers in, units_1 out
        self.fc2 = nn.Linear(units_1, units_2)   # <-- new: the second hidden layer takes in what fc1 gives out
        self.fc3 = nn.Linear(units_2, 1)         # <-- the last layer: one risk score out

    def forward(self, x):
        """Send a batch of students x through the layers and return one risk score per student."""
        x = torch.relu(self.fc1(x))              # <---- the first hidden layer, then the bend
        # <-- new: write one line here that sends x through self.fc2, then the bend (copy the line above and edit it)
        return self.fc3(x)                       # <---- no bend and no squeeze after the last layer

In [ ]:
# Part 2 (b): build one object from your class, train it, and plot it — run me, do not edit
torch.manual_seed(5001)                  # the same random starting weights every time you run this cell
my_net = MyNet()                         # <---- one object built from your class
print(my_net)                            # the layers my_net holds as attributes
my_history = train_network(my_net)       # <---- the same function as in Part 1, now with your object
plot_history(my_history, "MyNet: two hidden layers")

In [ ]:
#@title 🔎 Checkpoint helpers — run me, do not edit (Colab hides the code; you do not need to read it)
def checkpoint():
    """Check Part 2: the class MyNet, the object my_net, and its training record my_history."""
    cls = globals().get("MyNet")
    if not (isinstance(cls, type) and issubclass(cls, nn.Module)):
        print("❌ run the cell under (a) first: it defines the class MyNet")
        return
    doc = (cls.__doc__ or "").strip()
    has_doc = bool(doc) and "___" not in doc
    print("✅" if has_doc else "❌", "MyNet has a docstring" + ("" if has_doc else ": replace ___ with one sentence"))
    try:
        net = cls()
    except Exception as err:
        print(f"❌ MyNet() does not work yet: {type(err).__name__}: {err}")
        return
    layers = [(name, m) for name, m in net.named_modules() if isinstance(m, nn.Linear)]
    used = set()
    hooks = [m.register_forward_hook(lambda module, inputs, output, name=name: used.add(name)) for name, m in layers]
    try:
        with torch.no_grad():
            out = net(X_train[:5])
    except Exception as err:
        print(f"❌ MyNet does not work yet: {type(err).__name__}: {err}")
        return
    finally:
        for hook in hooks:
            hook.remove()
    if not isinstance(out, torch.Tensor):
        print("❌ forward should end with return self.fc3(x)")
        return
    print("✅" if tuple(out.shape) == (5, 1) else "❌",
          f"MyNet gives one risk score per student (shape {tuple(out.shape)}, expected (5, 1))")
    n_hidden = len(layers) - 1
    print("✅" if n_hidden >= 2 else "❌", f"MyNet has {n_hidden} hidden layer(s)" +
          ("" if n_hidden >= 2 else ": add a second hidden layer"))
    unused = ["self." + name for name, _ in layers if name not in used]
    print("✅" if not unused else "❌", "forward uses every layer" +
          ("" if not unused else f": {', '.join(unused)} is built in __init__ but never used in forward"))
    mine, history = globals().get("my_net"), globals().get("my_history")
    if not isinstance(mine, cls) or not (isinstance(history, dict) and history.get("valid_accuracy")):
        print("❌ run the cell under (b) again: it builds my_net from your latest MyNet and trains it")
        return
    final = history["valid_accuracy"][-1]
    if final > 0.70:
        print(f"✅ my_net learned: its final validation accuracy is {final}")
    elif abs(final - 0.348) < 0.03:
        print(f"❌ my_net's final validation accuracy is {final}, because every guess is 1: remove any bend or "
              "torch.sigmoid after the last layer, then run (a) and (b) again")
    else:
        print(f"❌ my_net's final validation accuracy is {final}: it hardly learned. Check forward, then run (a) and (b) again")

print("Checkpoint helpers ready ✅")

In [ ]:
# 🔎 Checkpoint — run me, do not edit
checkpoint()

---
## Part 3 — What did you build? (30 points)

The cell below puts your network next to TinyNet. It is also the first time you see your network's accuracy on the test students. Run it, then answer the four questions under it.

In [ ]:
# Part 3: TinyNet and your network side by side — run me, do not edit
comparison = pd.DataFrame([tiny_results, report("MyNet", my_net)])
comparison

Answer each question in a sentence or two in the cell below. Questions 1 and 2 are about the class, and questions 3 and 4 are about the network.

1. In `my_net = MyNet()`, which is the class and which is the object? Name one attribute of `my_net` and say what it holds.
2. What does `__init__` do in your class, and what does `forward` do? Why does `__init__` start with `super().__init__()`?
3. Compare your plot from Part 2 with TinyNet's plot from step 9. Which network memorized the training students more, and how can you tell from the two lines?
4. Look at the `weights` and `test_accuracy` columns. Did your bigger network do better than TinyNet on the test students? What does that tell you about bigger networks?

You may ask an AI tool to help you read the table or the plots. If you do, check one of its numbers against your own table and say that you did.

*Double-click this cell and write your answers here.*

---
## 📤 Submission

1. In Colab, run *Runtime ▸ Run all* and make sure every cell finishes **without errors**, with ✅ on every line of the checkpoint.
2. Download your notebook (*File ▸ Download ▸ Download .ipynb*) and upload it to Canvas by **Monday, Nov 23, 11:59 pm**.
3. If you used AI, also upload the draft you saved before you used it, as the syllabus asks.

**Code quality (10 points)** is about your class: a one-sentence docstring, a comment on the lines you added, and no placeholder left from the template.

---
**Data credit.** Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict Students' Dropout and Academic Success* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89. Shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes for this assignment: 13 of the 36 student columns kept, plus `Target`; program names, `daytime`/`evening`, and `female`/`male` written in place of number codes; first-semester grades rounded to 2 decimals; a `student_id` added; the students split at random into two halves with nearly the same share of dropouts; values separated by commas instead of semicolons.